# The Complete Guide to Attention Mechanisms: From Scratch to Transformers

**Understand and implement every major attention variant in PyTorch**

---

> **TL;DR** -- This notebook implements **every major attention mechanism from scratch in PyTorch**: Bahdanau (additive), Luong (dot/general/concat), Scaled Dot-Product Self-Attention, and Multi-Head Attention. We build a complete Transformer encoder, train it on a sequence reversal task, and visualize learned attention patterns across all layers and heads. Includes entropy analysis, scaling experiments, and a head-to-head comparison of attention variants on sequence tasks.

### **Key Results at a Glance**
| What You Build | Key Insight |
|:---|:---|
| Bahdanau Attention (2015) | First attention -- additive scoring with learned alignment |
| Luong Attention (3 variants) | Simpler dot-product scoring, more efficient |
| Scaled Dot-Product Self-Attention | Core of Transformers -- scaling by sqrt(d_k) is critical |
| Multi-Head Attention (8 heads) | Each head learns different relationship types |
| Full Transformer Encoder Block | LayerNorm + residuals + FFN = the modern workhorse |
| Training + Attention Visualization | See what the model actually learns to attend to |

### **Applicable Kaggle Competitions & Use Cases**
- [Feedback Prize - English Language Learning](https://www.kaggle.com/competitions/feedback-prize-english-language-learning) -- Transformer-based text scoring
- [LLM Science Exam](https://www.kaggle.com/competitions/kaggle-llm-science-exam) -- understanding how LLMs work
- [Google AI4Code](https://www.kaggle.com/competitions/AI4Code) -- code understanding with attention
- Any project using **BERT, GPT, or Vision Transformers**

---

If this notebook helps your understanding, please **upvote** -- it motivates more educational content!

## 📑 Table of Contents

1. [🔧 Setup & Imports](#1)
2. [🎯 Why Attention? The Intuition](#2)
3. [📝 Bahdanau (Additive) Attention](#3)
4. [✖️ Luong (Multiplicative) Attention](#4)
5. [🔄 Self-Attention (Scaled Dot-Product)](#5)
6. [👑 Multi-Head Attention](#6)
7. [🏗️ The Full Transformer Block](#7)
8. [📊 Attention Visualization Gallery](#8)
9. [🏁 Comparison on Sequence Tasks](#9)
10. [📝 Key Takeaways](#10)

<a id='1'></a>
## 1. 🔧 Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

<a id='2'></a>
## 2. 🎯 Why Attention? The Intuition

### The Bottleneck Problem

Traditional sequence-to-sequence models compress an entire input sequence into a **single fixed-size vector** (the encoder's final hidden state). This creates an information bottleneck — long sequences lose information.

```
Without Attention:
Input: [w1, w2, w3, ..., wn] ─▶ Encoder ─▶ [single vector] ─▶ Decoder ─▶ Output
                                            ↑ BOTTLENECK

With Attention:
Input: [w1, w2, w3, ..., wn] ─▶ Encoder ─▶ [h1, h2, h3, ..., hn]
                                                    │
                                            Attention weights
                                                    │
                                   Decoder ◀── weighted sum ──▶ Output
```

**Attention allows the decoder to look at ALL encoder hidden states**, focusing on the most relevant ones for each output step.

### The Three Key Concepts

All attention mechanisms share three core concepts:

- **Query (Q)**: What am I looking for? (decoder state)
- **Key (K)**: What information do I contain? (encoder states)
- **Value (V)**: What is my actual content? (encoder states)

The attention score between a query and key determines how much of the corresponding value to include:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\text{score}(Q, K)\right) \cdot V$$

Different attention variants differ in **how they compute the score function**.

### Synthetic Sequence Data

We'll create synthetic data for our experiments: a simple sequence reversal task and a sorting task.

In [ ]:
def generate_sequence_data(n_samples=1000, seq_len=10, vocab_size=20, task='reverse'):
    """Generate synthetic sequence-to-sequence data."""
    # Reserve 0 for padding, 1 for SOS, 2 for EOS
    src = torch.randint(3, vocab_size, (n_samples, seq_len))
    
    if task == 'reverse':
        tgt = src.flip(1)
    elif task == 'sort':
        tgt, _ = src.sort(dim=1)
    elif task == 'copy':
        tgt = src.clone()
    else:
        raise ValueError(f'Unknown task: {task}')
    
    return src, tgt

# Generate datasets
VOCAB_SIZE = 20
SEQ_LEN = 8
src_train, tgt_train = generate_sequence_data(2000, SEQ_LEN, VOCAB_SIZE, 'reverse')
src_test, tgt_test = generate_sequence_data(200, SEQ_LEN, VOCAB_SIZE, 'reverse')

print(f'Training: {src_train.shape}, Test: {src_test.shape}')
print(f'\nSample input:  {src_train[0].tolist()}')
print(f'Sample target: {tgt_train[0].tolist()} (reversed)')

<a id='3'></a>
## 3. 📝 Bahdanau (Additive) Attention

Proposed by Bahdanau et al. (2015), this was the **first attention mechanism** for neural machine translation.

### The Math

Given decoder hidden state $s_{t-1}$ and encoder hidden states $h_1, ..., h_n$:

$$e_{t,i} = v^T \tanh(W_s s_{t-1} + W_h h_i)$$

$$\alpha_{t,i} = \frac{\exp(e_{t,i})}{\sum_j \exp(e_{t,j})}$$

$$c_t = \sum_i \alpha_{t,i} h_i$$

Where:
- $e_{t,i}$ is the alignment score (energy)
- $\alpha_{t,i}$ are the attention weights (sum to 1)
- $c_t$ is the context vector (weighted sum of encoder states)
- $W_s$, $W_h$, $v$ are learned parameters

In [ ]:
class BahdanauAttention(nn.Module):
    """Bahdanau (Additive) Attention.
    
    Score = v^T * tanh(W_s * s + W_h * h)
    """
    
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_s = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_h = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
    
    def forward(self, decoder_state, encoder_outputs):
        """
        Args:
            decoder_state: (batch, hidden_dim)
            encoder_outputs: (batch, seq_len, hidden_dim)
        Returns:
            context: (batch, hidden_dim)
            weights: (batch, seq_len)
        """
        # decoder_state: (batch, 1, hidden_dim) for broadcasting
        s = self.W_s(decoder_state.unsqueeze(1))  # (batch, 1, hidden)
        h = self.W_h(encoder_outputs)              # (batch, seq_len, hidden)
        
        # Alignment scores
        energy = self.v(torch.tanh(s + h))  # (batch, seq_len, 1)
        energy = energy.squeeze(-1)          # (batch, seq_len)
        
        # Attention weights
        weights = F.softmax(energy, dim=-1)  # (batch, seq_len)
        
        # Context vector
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs)  # (batch, 1, hidden)
        context = context.squeeze(1)  # (batch, hidden)
        
        return context, weights


# Quick test
hidden_dim = 64
batch_size = 4
seq_len = 8

attn = BahdanauAttention(hidden_dim)
dec_state = torch.randn(batch_size, hidden_dim)
enc_outputs = torch.randn(batch_size, seq_len, hidden_dim)

context, weights = attn(dec_state, enc_outputs)
print(f'Decoder state shape:   {dec_state.shape}')
print(f'Encoder outputs shape: {enc_outputs.shape}')
print(f'Context vector shape:  {context.shape}')
print(f'Attention weights shape: {weights.shape}')
print(f'Weights sum: {weights[0].sum().item():.4f} (should be 1.0)')
print(f'\nSample weights: {weights[0].detach().numpy().round(3)}')

> **Key Takeaway -- Bahdanau Attention:** This was the breakthrough that eliminated the information bottleneck in seq2seq models. The key insight is learning *alignment* between decoder and encoder positions. The additive scoring function (`v^T * tanh(W_s * s + W_h * h)`) is more expressive than dot-product but slower. Use this when you need maximum flexibility in scoring.

<a id='4'></a>
## 4. ✖️ Luong (Multiplicative) Attention

Luong et al. (2015) proposed simpler attention variants that are **more computationally efficient**.

### Three Scoring Functions

**Dot Product:**
$$\text{score}(s_t, h_i) = s_t^T h_i$$

**General (Bilinear):**
$$\text{score}(s_t, h_i) = s_t^T W_a h_i$$

**Concat (similar to Bahdanau):**
$$\text{score}(s_t, h_i) = v^T \tanh(W_a [s_t; h_i])$$

The dot product variant requires $s_t$ and $h_i$ to have the same dimension, while the general variant uses a learned weight matrix to handle different dimensions.

In [ ]:
class LuongAttention(nn.Module):
    """Luong (Multiplicative) Attention with multiple scoring functions."""
    
    def __init__(self, hidden_dim, method='general'):
        super().__init__()
        self.method = method
        self.hidden_dim = hidden_dim
        
        if method == 'general':
            self.W = nn.Linear(hidden_dim, hidden_dim, bias=False)
        elif method == 'concat':
            self.W = nn.Linear(hidden_dim * 2, hidden_dim, bias=False)
            self.v = nn.Linear(hidden_dim, 1, bias=False)
    
    def forward(self, decoder_state, encoder_outputs):
        """
        Args:
            decoder_state: (batch, hidden_dim)
            encoder_outputs: (batch, seq_len, hidden_dim)
        Returns:
            context: (batch, hidden_dim)
            weights: (batch, seq_len)
        """
        if self.method == 'dot':
            # score = s^T * h
            energy = torch.bmm(encoder_outputs, 
                               decoder_state.unsqueeze(-1))  # (batch, seq_len, 1)
        elif self.method == 'general':
            # score = s^T * W * h
            energy = torch.bmm(self.W(encoder_outputs), 
                               decoder_state.unsqueeze(-1))  # (batch, seq_len, 1)
        elif self.method == 'concat':
            # score = v^T * tanh(W * [s; h])
            s_expanded = decoder_state.unsqueeze(1).expand_as(encoder_outputs)
            combined = torch.cat([s_expanded, encoder_outputs], dim=-1)
            energy = self.v(torch.tanh(self.W(combined)))  # (batch, seq_len, 1)
        
        energy = energy.squeeze(-1)  # (batch, seq_len)
        weights = F.softmax(energy, dim=-1)
        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)
        
        return context, weights


# Compare the three Luong variants
print('Luong Attention Variants:')
print('=' * 50)
for method in ['dot', 'general', 'concat']:
    attn = LuongAttention(hidden_dim, method=method)
    context, weights = attn(dec_state, enc_outputs)
    n_params = sum(p.numel() for p in attn.parameters())
    print(f'\n{method.upper():>10}: params={n_params:,}, weights={weights[0].detach().numpy().round(3)}')

### Comparing Attention Patterns

Let's visualize how Bahdanau and Luong attention produce different weight distributions on the same input.

> **Key Takeaway -- Luong vs Bahdanau:** Luong's dot-product attention is faster and often works just as well. The general (bilinear) variant adds a learnable matrix for minimal cost. In practice, the **scaled dot-product** variant (used in Transformers) became the dominant approach because it parallelizes perfectly.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

attention_variants = [
    ('Bahdanau', BahdanauAttention(hidden_dim)),
    ('Luong (dot)', LuongAttention(hidden_dim, 'dot')),
    ('Luong (general)', LuongAttention(hidden_dim, 'general')),
    ('Luong (concat)', LuongAttention(hidden_dim, 'concat')),
]

# Generate attention weights for multiple decoder steps
n_dec_steps = 8
torch.manual_seed(42)
enc_out = torch.randn(1, seq_len, hidden_dim)

for ax, (name, attn_module) in zip(axes, attention_variants):
    weight_matrix = []
    for step in range(n_dec_steps):
        dec_s = torch.randn(1, hidden_dim)
        _, w = attn_module(dec_s, enc_out)
        weight_matrix.append(w[0].detach().numpy())
    
    weight_matrix = np.array(weight_matrix)
    sns.heatmap(weight_matrix, ax=ax, cmap='YlOrRd', vmin=0, vmax=0.5,
                xticklabels=[f'enc_{i}' for i in range(seq_len)],
                yticklabels=[f'dec_{i}' for i in range(n_dec_steps)],
                cbar_kws={'shrink': 0.8})
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Encoder Position')
    if ax == axes[0]:
        ax.set_ylabel('Decoder Step')

plt.suptitle('Attention Weight Patterns by Variant', fontsize=15, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

<a id='5'></a>
## 5. 🔄 Self-Attention (Scaled Dot-Product)

Self-attention allows each position in a sequence to attend to **all other positions in the same sequence**. This is the core mechanism of the Transformer.

### The Math

Given input $X \in \mathbb{R}^{n \times d}$:

$$Q = XW^Q, \quad K = XW^K, \quad V = XW^V$$

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

The scaling factor $\sqrt{d_k}$ prevents the dot products from growing too large, which would push softmax into regions with very small gradients.

### Why Scaling Matters

For vectors of dimension $d_k$, the expected value of the dot product is 0 with variance $d_k$. Without scaling:
- Large $d_k$ $\rightarrow$ large dot products $\rightarrow$ peaked softmax $\rightarrow$ vanishing gradients
- Scaling by $\sqrt{d_k}$ normalizes the variance back to 1

In [ ]:
class ScaledDotProductAttention(nn.Module):
    """Scaled Dot-Product Self-Attention.
    
    Attention(Q, K, V) = softmax(Q @ K^T / sqrt(d_k)) @ V
    """
    
    def __init__(self, d_model, d_k=None, d_v=None):
        super().__init__()
        self.d_k = d_k or d_model
        self.d_v = d_v or d_model
        
        self.W_q = nn.Linear(d_model, self.d_k)
        self.W_k = nn.Linear(d_model, self.d_k)
        self.W_v = nn.Linear(d_model, self.d_v)
        self.scale = math.sqrt(self.d_k)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: optional (batch, seq_len, seq_len) or (1, seq_len, seq_len)
        Returns:
            output: (batch, seq_len, d_v)
            weights: (batch, seq_len, seq_len)
        """
        Q = self.W_q(x)  # (batch, seq_len, d_k)
        K = self.W_k(x)  # (batch, seq_len, d_k)
        V = self.W_v(x)  # (batch, seq_len, d_v)
        
        # Compute attention scores
        scores = torch.bmm(Q, K.transpose(1, 2)) / self.scale  # (batch, seq_len, seq_len)
        
        # Apply mask (e.g., for causal/autoregressive attention)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        weights = F.softmax(scores, dim=-1)  # (batch, seq_len, seq_len)
        
        # Weighted sum of values
        output = torch.bmm(weights, V)  # (batch, seq_len, d_v)
        
        return output, weights


# Test self-attention
d_model = 64
self_attn = ScaledDotProductAttention(d_model)

x = torch.randn(batch_size, seq_len, d_model)
out, weights = self_attn(x)

print(f'Input shape:   {x.shape}')
print(f'Output shape:  {out.shape}')
print(f'Weights shape: {weights.shape}')
print(f'\nEach position attends to all {seq_len} positions.')
print(f'Weights sum per row: {weights[0].sum(dim=-1).detach().numpy().round(4)}')

### Visualizing Self-Attention

In self-attention, every position attends to every other position. Let's visualize the attention matrix.

In [ ]:
# Create a more interpretable example with token labels
tokens = ['The', 'cat', 'sat', 'on', 'the', 'mat', 'and', 'purred']

torch.manual_seed(123)
x = torch.randn(1, len(tokens), d_model)
_, attn_weights = self_attn(x)
w = attn_weights[0].detach().numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Full attention
sns.heatmap(w, ax=axes[0], cmap='Blues', annot=True, fmt='.2f',
            xticklabels=tokens, yticklabels=tokens,
            cbar_kws={'shrink': 0.8})
axes[0].set_title('Self-Attention Weights (Full)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Key Position (attended to)')
axes[0].set_ylabel('Query Position (attending from)')

# Causal mask
causal_mask = torch.tril(torch.ones(len(tokens), len(tokens))).unsqueeze(0)
_, causal_weights = self_attn(x, mask=causal_mask)
cw = causal_weights[0].detach().numpy()

sns.heatmap(cw, ax=axes[1], cmap='Blues', annot=True, fmt='.2f',
            xticklabels=tokens, yticklabels=tokens,
            cbar_kws={'shrink': 0.8})
axes[1].set_title('Causal Self-Attention (Masked)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Key Position (attended to)')
axes[1].set_ylabel('Query Position (attending from)')

plt.suptitle('Self-Attention: Full vs Causal Masking', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('In causal attention, each position can only attend to earlier positions.')
print('This is used in decoder/autoregressive models like GPT.')

> **Key Takeaway -- Scaling Factor:** Without the `1/sqrt(d_k)` scaling, attention scores have variance proportional to `d_k`. At d_k=512, unscaled scores have ~100x the variance of scaled scores, causing softmax to saturate and gradients to vanish. **This single insight made deep Transformers trainable.** Always scale your dot products!

### Why Scaling by $\sqrt{d_k}$ Matters

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, dk in zip(axes, [8, 64, 512]):
    torch.manual_seed(42)
    q = torch.randn(1, 10, dk)
    k = torch.randn(1, 10, dk)
    
    # Unscaled
    raw_scores = torch.bmm(q, k.transpose(1, 2))[0].detach().numpy().flatten()
    # Scaled
    scaled_scores = (torch.bmm(q, k.transpose(1, 2)) / math.sqrt(dk))[0].detach().numpy().flatten()
    
    ax.hist(raw_scores, bins=30, alpha=0.6, label=f'Unscaled (var={np.var(raw_scores):.1f})', color='#e74c3c')
    ax.hist(scaled_scores, bins=30, alpha=0.6, label=f'Scaled (var={np.var(scaled_scores):.2f})', color='#3498db')
    ax.set_title(f'd_k = {dk}', fontsize=13, fontweight='bold')
    ax.set_xlabel('Score Value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=10)

plt.suptitle('Effect of Scaling on Score Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print('Without scaling, larger d_k produces scores with higher variance,')
print('which makes softmax extremely peaked and gradients vanishingly small.')

<a id='6'></a>
## 6. 👑 Multi-Head Attention

Instead of performing a single attention function, multi-head attention runs **h parallel attention heads**, each learning to attend to different types of relationships.

### The Math

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h) W^O$$

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

Where $W_i^Q \in \mathbb{R}^{d_{model} \times d_k}$, $W_i^K \in \mathbb{R}^{d_{model} \times d_k}$, $W_i^V \in \mathbb{R}^{d_{model} \times d_v}$

With $d_k = d_v = d_{model} / h$, the computational cost is similar to single-head attention with full dimensionality.

### Why Multiple Heads?

Different heads can learn to attend to:
- **Positional patterns** (nearby tokens)
- **Syntactic relationships** (subject-verb agreement)
- **Semantic relationships** (coreference, topic)
- **Rare but important connections** (long-distance dependencies)

> **Key Takeaway -- Multi-Head Attention:** Multiple heads are not just redundancy -- each head specializes in different types of relationships (local context, global structure, syntactic patterns). The output projection (`W_o`) learns to combine these complementary views. Using 8-16 heads is standard; fewer can hurt quality, but more rarely helps beyond 16.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-Head Self-Attention.
    
    Runs h attention heads in parallel, each with d_k = d_model / h.
    """
    
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, 'd_model must be divisible by n_heads'
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        # Linear projections for Q, K, V (all heads combined)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.d_k)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch, seq_len, d_model)
            mask: optional attention mask
        Returns:
            output: (batch, seq_len, d_model)
            weights: (batch, n_heads, seq_len, seq_len)
        """
        batch_size, seq_len, _ = x.shape
        
        # Project and reshape to (batch, n_heads, seq_len, d_k)
        Q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention for all heads simultaneously
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale  # (batch, n_heads, seq, seq)
        
        if mask is not None:
            if mask.dim() == 3:
                mask = mask.unsqueeze(1)  # (batch, 1, seq, seq) broadcasts over heads
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        weights = F.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        
        # Apply attention to values
        attn_output = torch.matmul(weights, V)  # (batch, n_heads, seq_len, d_k)
        
        # Concatenate heads: (batch, seq_len, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        # Final linear projection
        output = self.W_o(attn_output)
        
        return output, weights


# Test multi-head attention
mha = MultiHeadAttention(d_model=64, n_heads=8)
x = torch.randn(batch_size, seq_len, 64)
out, weights = mha(x)

n_params = sum(p.numel() for p in mha.parameters())
print(f'Input:   {x.shape}')
print(f'Output:  {out.shape}')
print(f'Weights: {weights.shape}  (batch, heads, seq, seq)')
print(f'Parameters: {n_params:,}')
print(f'd_k per head: {mha.d_k}')

### Multi-Head Attention Visualization

Each head learns different attention patterns. Let's visualize all 8 heads.

In [ ]:
torch.manual_seed(42)
mha_viz = MultiHeadAttention(d_model=64, n_heads=8)
x_viz = torch.randn(1, len(tokens), 64)
_, mha_weights = mha_viz(x_viz)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for head_idx in range(8):
    w = mha_weights[0, head_idx].detach().numpy()
    sns.heatmap(w, ax=axes[head_idx], cmap='viridis', vmin=0,
                xticklabels=tokens, yticklabels=tokens if head_idx % 4 == 0 else False,
                cbar=False)
    axes[head_idx].set_title(f'Head {head_idx + 1}', fontsize=12, fontweight='bold')

plt.suptitle('All 8 Attention Heads - Each Learns Different Patterns', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Notice how different heads learn different attention patterns:')
print('- Some focus on nearby positions (local patterns)')
print('- Some attend broadly (global patterns)')
print('- Some show more peaked distributions (specific relationships)')

<a id='7'></a>
## 7. 🏗️ The Full Transformer Block

A Transformer encoder block combines:
1. Multi-head self-attention
2. Layer normalization
3. Feed-forward network (FFN)
4. Residual connections

```
Input
  │
  ├──────────────┐
  │              │
  │     Multi-Head Attention
  │              │
  └───▶ Add & LayerNorm
         │
         ├───────────┐
         │           │
         │     Feed-Forward
         │           │
         └──▶ Add & LayerNorm
                │
              Output
```

### Positional Encoding

Since attention is permutation-invariant (it doesn't know token order), we inject positional information:

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding from 'Attention Is All You Need'."""
    
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network."""
    
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)


class TransformerEncoderBlock(nn.Module):
    """Single Transformer encoder block."""
    
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Multi-head attention with residual connection
        attn_out, weights = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_out))
        
        # Feed-forward with residual connection
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        
        return x, weights


# Test the full block
block = TransformerEncoderBlock(d_model=64, n_heads=8)
x = torch.randn(batch_size, seq_len, 64)
out, weights = block(x)

n_params = sum(p.numel() for p in block.parameters())
print(f'TransformerEncoderBlock:')
print(f'  Input:      {x.shape}')
print(f'  Output:     {out.shape}')
print(f'  Parameters: {n_params:,}')
print(f'  Components: MultiHeadAttention + LayerNorm + FeedForward + LayerNorm')

### Positional Encoding Visualization

In [ ]:
pe = PositionalEncoding(d_model=64, max_len=100)
pe_values = pe.pe[0, :50, :64].numpy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
im = axes[0].imshow(pe_values.T, aspect='auto', cmap='RdBu_r', interpolation='nearest')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Dimension')
axes[0].set_title('Positional Encoding Heatmap', fontsize=13, fontweight='bold')
plt.colorbar(im, ax=axes[0])

# Individual dimensions
for dim in [0, 1, 4, 5, 10, 11]:
    axes[1].plot(pe_values[:, dim], label=f'dim {dim}', alpha=0.8)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Value')
axes[1].set_title('Positional Encoding by Dimension', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()
print('Low dimensions oscillate quickly (fine position), high dimensions oscillate slowly (global position).')

### Complete Transformer Encoder Model

Let's stack multiple blocks into a full encoder for sequence transduction.

In [ ]:
class TransformerEncoder(nn.Module):
    """Full Transformer Encoder model."""
    
    def __init__(self, vocab_size, d_model=64, n_heads=4, n_layers=3, 
                 d_ff=256, max_len=100, dropout=0.1, n_classes=None):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])
        
        self.norm = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, vocab_size)
    
    def forward(self, x, mask=None):
        # Embed and add positional encoding
        x = self.embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        
        all_weights = []
        for layer in self.layers:
            x, weights = layer(x, mask)
            all_weights.append(weights)
        
        x = self.norm(x)
        logits = self.output_proj(x)
        
        return logits, all_weights


model = TransformerEncoder(
    vocab_size=VOCAB_SIZE, d_model=64, n_heads=4, 
    n_layers=3, d_ff=128, max_len=50
)

n_params = sum(p.numel() for p in model.parameters())
print(f'Full Transformer Encoder:')
print(f'  Vocab size: {VOCAB_SIZE}')
print(f'  d_model: 64, n_heads: 4, n_layers: 3')
print(f'  Total parameters: {n_params:,}')

# Forward pass
sample = src_train[:4]
logits, all_weights = model(sample)
print(f'\n  Input:  {sample.shape}')
print(f'  Logits: {logits.shape}')
print(f'  Attention weights: {len(all_weights)} layers x {all_weights[0].shape}')

model.eval()
with torch.no_grad():
    sample = src_test[:1]
    logits, all_weights = model(sample)
    preds = logits.argmax(dim=-1)

print(f'Input:      {sample[0].tolist()}')
print(f'Target:     {tgt_test[0].tolist()}')
print(f'Prediction: {preds[0].tolist()}')
print(f'Correct:    {(preds[0] == tgt_test[0]).all().item()}')

# Visualize attention from all layers
fig, axes = plt.subplots(3, 4, figsize=(20, 14))
input_tokens = [str(t) for t in sample[0].tolist()]

for layer_idx in range(3):
    for head_idx in range(4):
        w = all_weights[layer_idx][0, head_idx].detach().numpy()
        ax = axes[layer_idx, head_idx]
        sns.heatmap(w, ax=ax, cmap='Blues', vmin=0,
                    xticklabels=input_tokens, yticklabels=input_tokens if head_idx == 0 else False,
                    cbar=False, annot=True, fmt='.2f', annot_kws={'size': 8})
        ax.set_title(f'Layer {layer_idx+1}, Head {head_idx+1}', fontsize=11, fontweight='bold')
        if head_idx == 0:
            ax.set_ylabel('Query', fontsize=10)
        if layer_idx == 2:
            ax.set_xlabel('Key', fontsize=10)

plt.suptitle('Learned Attention Patterns Across All Layers and Heads', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def train_model(model, src, tgt, epochs=30, lr=1e-3, batch_size=64):
    """Train the model and return loss history."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    model.train()
    losses = []
    accs = []
    
    n_samples = src.shape[0]
    for epoch in range(epochs):
        epoch_loss = 0
        n_correct = 0
        n_total = 0
        
        # Shuffle
        perm = torch.randperm(n_samples)
        src_shuffled = src[perm]
        tgt_shuffled = tgt[perm]
        
        for i in range(0, n_samples, batch_size):
            batch_src = src_shuffled[i:i+batch_size]
            batch_tgt = tgt_shuffled[i:i+batch_size]
            
            logits, _ = model(batch_src)
            loss = criterion(logits.view(-1, VOCAB_SIZE), batch_tgt.view(-1))
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            preds = logits.argmax(dim=-1)
            n_correct += (preds == batch_tgt).sum().item()
            n_total += batch_tgt.numel()
        
        avg_loss = epoch_loss / (n_samples // batch_size)
        acc = n_correct / n_total
        losses.append(avg_loss)
        accs.append(acc)
        
        if (epoch + 1) % 5 == 0:
            print(f'Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | Acc: {acc:.4f}')
    
    return losses, accs


# Train!
print('Training Transformer on sequence reversal task...')
print('=' * 50)
model = TransformerEncoder(vocab_size=VOCAB_SIZE, d_model=64, n_heads=4, n_layers=3, d_ff=128)
losses, accs = train_model(model, src_train, tgt_train, epochs=30)

In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(losses, color='#e74c3c', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss', fontsize=13, fontweight='bold')

ax2.plot(accs, color='#2ecc71', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Token Accuracy')
ax2.set_title('Training Accuracy', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 1)

plt.suptitle('Transformer Training on Sequence Reversal', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Learned Attention Patterns

After training, let's visualize what the model has learned. For a reversal task, we'd expect some heads to learn anti-diagonal patterns (mapping position i to position n-i).

In [ ]:
model.set_mode = None  # ensure no training mode side effects
with torch.no_grad():
    sample = src_test[:1]
    logits, all_weights = model(sample)
    preds = logits.argmax(dim=-1)

print(f'Input:      {sample[0].tolist()}')
print(f'Target:     {tgt_test[0].tolist()}')
print(f'Prediction: {preds[0].tolist()}')
print(f'Correct:    {(preds[0] == tgt_test[0]).all().item()}')

# Visualize attention from all layers
fig, axes = plt.subplots(3, 4, figsize=(20, 14))
input_tokens = [str(t) for t in sample[0].tolist()]

for layer_idx in range(3):
    for head_idx in range(4):
        w = all_weights[layer_idx][0, head_idx].detach().numpy()
        ax = axes[layer_idx, head_idx]
        sns.heatmap(w, ax=ax, cmap='Blues', vmin=0,
                    xticklabels=input_tokens, yticklabels=input_tokens if head_idx == 0 else False,
                    cbar=False, annot=True, fmt='.2f', annot_kws={'size': 8})
        ax.set_title(f'Layer {layer_idx+1}, Head {head_idx+1}', fontsize=11, fontweight='bold')
        if head_idx == 0:
            ax.set_ylabel('Query', fontsize=10)
        if layer_idx == 2:
            ax.set_xlabel('Key', fontsize=10)

plt.suptitle('Learned Attention Patterns Across All Layers and Heads', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Attention Entropy Analysis

Entropy tells us how spread out the attention is. Low entropy = focused on few positions. High entropy = attending broadly.

In [ ]:
def attention_entropy(weights):
    """Compute entropy of attention weights."""
    # Clamp to avoid log(0)
    w = weights.clamp(min=1e-10)
    entropy = -(w * w.log()).sum(dim=-1)
    return entropy

with torch.no_grad():
    _, all_w = model(src_test[:100])

fig, ax = plt.subplots(figsize=(12, 5))
max_entropy = math.log(SEQ_LEN)

for layer_idx, w in enumerate(all_w):
    for head_idx in range(w.shape[1]):
        ent = attention_entropy(w[:, head_idx]).mean().item()
        ax.bar(layer_idx * 5 + head_idx, ent / max_entropy, 
               color=plt.cm.tab10(head_idx / 4), width=0.8,
               label=f'Head {head_idx+1}' if layer_idx == 0 else '')

ax.set_xticks([2, 7, 12])
ax.set_xticklabels(['Layer 1', 'Layer 2', 'Layer 3'])
ax.set_ylabel('Normalized Entropy')
ax.set_title('Attention Entropy by Layer and Head', fontsize=14, fontweight='bold')
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='Max (uniform)')
ax.legend()
plt.tight_layout()
plt.show()

print('Lower entropy = more focused attention (attends to specific positions)')
print('Higher entropy = more diffuse attention (attends broadly)')

<a id='9'></a>
## 9. 🏁 Comparison on Sequence Tasks

Let's compare different attention mechanisms on our sequence tasks.

In [ ]:
<a id='10'></a>
## 10. Key Takeaways

### What We Built

We implemented **every major attention mechanism** from scratch:

| Mechanism | Year | Score Function | Key Innovation |
|-----------|------|---------------|----------------|
| Bahdanau | 2015 | $v^T \tanh(W_s s + W_h h)$ | First attention for NMT |
| Luong (dot) | 2015 | $s^T h$ | Simplest, fastest |
| Luong (general) | 2015 | $s^T W h$ | Learnable, different dims |
| Self-Attention | 2017 | $\frac{QK^T}{\sqrt{d_k}}$ | Intra-sequence |
| Multi-Head | 2017 | h parallel heads | Multiple relationship types |

### Key Lessons

1. **Attention solves the bottleneck problem** -- allowing the decoder to access all encoder states rather than just the final one.

2. **Scaling matters** -- without $\sqrt{d_k}$ scaling, softmax becomes too peaked for large dimensions.

3. **Multiple heads capture different relationships** -- some learn local patterns, others learn global dependencies.

4. **Attention is interpretable** -- we can visualize what the model looks at, providing insight into its reasoning.

5. **Transformers outperform RNNs with attention** -- self-attention enables parallelization and captures long-range dependencies better.

### The Attention Evolution

```
2015: Bahdanau Attention (seq2seq with RNNs)
  |
2015: Luong Attention (simpler scoring)
  |
2017: Self-Attention + Transformer ("Attention Is All You Need")
  |
2018: BERT (bidirectional), GPT (autoregressive)
  |
2020+: Efficient attention (Linear, Sparse, Flash Attention)
  |
2023+: State-space models (Mamba) challenging attention
```

---

## Further Reading

**Foundational Papers:**
- [Attention Is All You Need](https://arxiv.org/abs/1706.03762) (Vaswani et al., 2017) -- the Transformer paper
- [Neural Machine Translation by Jointly Learning to Align and Translate](https://arxiv.org/abs/1409.0473) (Bahdanau et al., 2015) -- first attention
- [Effective Approaches to Attention-based NMT](https://arxiv.org/abs/1508.04025) (Luong et al., 2015) -- multiplicative attention

**Modern Extensions:**
- [FlashAttention](https://arxiv.org/abs/2205.14135) (Dao et al., 2022) -- IO-aware exact attention
- [Mamba](https://arxiv.org/abs/2312.00752) (Gu & Dao, 2023) -- selective state spaces as an alternative
- [Multi-Query Attention](https://arxiv.org/abs/1911.02150) -- efficient inference

**Kaggle Resources:**
- [Feedback Prize competitions](https://www.kaggle.com/competitions?search=feedback+prize) -- Transformer-based NLP
- [HuggingFace Transformers documentation](https://huggingface.co/docs/transformers) -- using pre-trained attention models

**Related Notebooks:**
- Check out my [RAG from Scratch](https://www.kaggle.com/lorenzoscaturchio) notebook to see retrieval augmented generation built step by step
- Check out my [Feature Engineering Cookbook](https://www.kaggle.com/lorenzoscaturchio) for 50 competition-winning techniques

---

### **If this notebook helped you understand attention, please upvote! Your support helps the Kaggle community discover educational content.**

Experiment ideas:
- Try different sequence tasks (sorting, copying, arithmetic)
- Implement Flash Attention for efficiency
- Add cross-attention for full encoder-decoder Transformer
- Implement sparse attention patterns

**Connect with me** for more ML educational content!

In [ ]:
# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#95a5a6', '#e74c3c', '#3498db', '#2ecc71']

for (name, data), color in zip(results.items(), colors):
    ax1.plot(data['train'], label=name, color=color, linewidth=2)
    ax2.plot(data['test'], label=name, color=color, linewidth=2)

ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Training Accuracy', fontsize=13, fontweight='bold')
ax1.legend()

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Test Accuracy', fontsize=13, fontweight='bold')
ax2.legend()

for ax in [ax1, ax2]:
    ax.set_ylim(0, 1)

plt.suptitle('Attention Mechanism Comparison: Sequence Reversal Task', 
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Parameter Count and Computational Complexity

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

models_info = [
    ('No Attention', SimpleSeq2SeqWithAttention(VOCAB_SIZE, 64, 'none')),
    ('Bahdanau', SimpleSeq2SeqWithAttention(VOCAB_SIZE, 64, 'bahdanau')),
    ('Luong (dot)', SimpleSeq2SeqWithAttention(VOCAB_SIZE, 64, 'luong_dot')),
    ('Luong (general)', SimpleSeq2SeqWithAttention(VOCAB_SIZE, 64, 'luong_general')),
    ('Transformer (3L)', TransformerEncoder(VOCAB_SIZE, 64, 4, 3, 128)),
]

names = [m[0] for m in models_info]
params = [sum(p.numel() for p in m[1].parameters()) for m in models_info]
test_scores = [results.get(n, {}).get('test', [0])[-1] for n in names[:-1]] + [accs[-1]]

scatter = ax.scatter(params, test_scores, s=200, c=colors + ['#9b59b6'], 
                     edgecolors='white', linewidth=2, zorder=5)

for name, p, s in zip(names, params, test_scores):
    ax.annotate(name, (p, s), textcoords='offset points', xytext=(10, 10),
                fontsize=11, fontweight='bold')

ax.set_xlabel('Number of Parameters', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('Accuracy vs Model Size', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

<a id='10'></a>
## 10. 📝 Key Takeaways

### What We Built

We implemented **every major attention mechanism** from scratch:

| Mechanism | Year | Score Function | Key Innovation |
|-----------|------|---------------|----------------|
| Bahdanau | 2015 | $v^T \tanh(W_s s + W_h h)$ | First attention for NMT |
| Luong (dot) | 2015 | $s^T h$ | Simplest, fastest |
| Luong (general) | 2015 | $s^T W h$ | Learnable, different dims |
| Self-Attention | 2017 | $\frac{QK^T}{\sqrt{d_k}}$ | Intra-sequence |
| Multi-Head | 2017 | h parallel heads | Multiple relationship types |

### Key Lessons

1. **Attention solves the bottleneck problem** — allowing the decoder to access all encoder states rather than just the final one.

2. **Scaling matters** — without $\sqrt{d_k}$ scaling, softmax becomes too peaked for large dimensions.

3. **Multiple heads capture different relationships** — some learn local patterns, others learn global dependencies.

4. **Attention is interpretable** — we can visualize what the model looks at, providing insight into its reasoning.

5. **Transformers outperform RNNs with attention** — self-attention enables parallelization and captures long-range dependencies better.

### The Attention Evolution

```
2015: Bahdanau Attention (seq2seq with RNNs)
  |
2015: Luong Attention (simpler scoring)
  |
2017: Self-Attention + Transformer ("Attention Is All You Need")
  |
2018: BERT (bidirectional), GPT (autoregressive)
  |
2020+: Efficient attention (Linear, Sparse, Flash Attention)
  |
2023+: State-space models (Mamba) challenging attention
```

---

**If this notebook helped you understand attention, please upvote!** 🙏

Experiment ideas:
- Try different sequence tasks (sorting, copying, arithmetic)
- Implement Flash Attention for efficiency
- Add cross-attention for full encoder-decoder Transformer
- Implement sparse attention patterns